# 03 Búsqueda vectorial con TF-IDF

**Nombre:** David Pilataxi  
**Fecha:** 11/11/2026

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

#### Resultado esperado — Sección 1

- La tabla tendrá **18 filas** (los 18 libros de Gutenberg incluidos en NLTK), una por documento.
- **Tamaño del vocabulario:** decenas de miles de términos (se espera entre 30 000 y 50 000).
- **Longitud de documentos (tokens procesados):** el mínimo corresponderá a un texto corto (poemas de Blake o una obra breve de Shakespeare, unos pocos miles de tokens); el máximo, a la Biblia (cientos de miles de tokens); la media quedará muy por debajo del máximo por la gran asimetría.
- Los libros extensos y de vocabulario variado (Moby Dick, Leaves of Grass, la Biblia) tendrán más de 10 000 términos únicos; los textos cortos (Blake, Buster Bear), poco más de mil.

In [1]:
%pip install -q nltk   # instala nltk si no está en tu entorno
import re, numpy as np, pandas as pd
import nltk
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

try: gutenberg.fileids()                       # descarga el corpus solo si falta
except LookupError: nltk.download("gutenberg")

# Preprocesamiento: minúsculas, solo letras, sin stopwords y sin tokens de <3 letras
def prep(texto):
    return [w for w in re.findall(r"[a-z]+", texto.lower())
            if w not in ENGLISH_STOP_WORDS and len(w) > 2]

ids    = gutenberg.fileids()                   # identificadores de documento
raw    = [gutenberg.raw(i) for i in ids]       # texto original (para fragmentos)
docs   = [prep(t) for t in raw]                # mismos documentos procesados para TODO
titulos = {"austen-emma.txt":"Emma", "austen-persuasion.txt":"Persuasion", "austen-sense.txt":"Sense and Sensibility",
 "bible-kjv.txt":"The King James Bible", "blake-poems.txt":"Poems (Blake)", "bryant-stories.txt":"Stories to Tell to Children",
 "burgess-busterbrown.txt":"The Adventures of Buster Bear", "carroll-alice.txt":"Alice's Adventures in Wonderland",
 "chesterton-ball.txt":"The Ball and the Cross", "chesterton-brown.txt":"The Wisdom of Father Brown",
 "chesterton-thursday.txt":"The Man Who Was Thursday", "edgeworth-parents.txt":"The Parent's Assistant",
 "melville-moby_dick.txt":"Moby Dick", "milton-paradise.txt":"Paradise Lost", "shakespeare-caesar.txt":"Julius Caesar",
 "shakespeare-hamlet.txt":"Hamlet", "shakespeare-macbeth.txt":"Macbeth", "whitman-leaves.txt":"Leaves of Grass"}
tit = [titulos[i] for i in ids]

vocab = sorted(set(w for d in docs for w in d))   # vocabulario con orden definido (alfabético)
lens  = [len(d) for d in docs]
display(pd.DataFrame({"Identificador": ids, "Título": tit, "Términos únicos": [len(set(d)) for d in docs]}))
print("Documentos:", len(docs), "| Vocabulario:", len(vocab))
print("Longitud mín/media/máx:", min(lens), round(np.mean(lens), 1), max(lens))

Note: you may need to restart the kernel to use updated packages.


,Identificador,Título,Términos únicos
0,austen-emma.txt,Emma,6787
1,austen-persuasion.txt,Persuasion,5443
2,austen-sense.txt,Sense and Sensibility,5990
3,bible-kjv.txt,The King James Bible,12290
4,blake-poems.txt,Poems (Blake),1328
5,bryant-stories.txt,Stories to Tell to Children,3556
6,burgess-busterbrown.txt,The Adventures of Buster Bear,1266
7,carroll-alice.txt,Alice's Adventures in Wonderland,2298
8,chesterton-ball.txt,The Ball and the Cross,7855
9,chesterton-brown.txt,The Wisdom of Father Brown,7423


Documentos: 18 | Vocabulario: 41071
Longitud mín/media/máx: 3536 49592.3 329877


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

#### Resultado esperado — Sección 2

Prueba manual con el mini-corpus de 3 documentos (N = 3): vocabulario ordenado `ave, gato, perro, pez`.

| término | DF | IDF = ln(3/DF)+1 | TF doc0 | TF-IDF doc0 |
|---|---|---|---|---|
| ave | 1 | 2.0986 | 0 | 0 |
| gato | 2 | 1.4055 | 2 | 2.8109 |
| perro | 2 | 1.4055 | 1 | 1.4055 |
| pez | 2 | 1.4055 | 0 | 0 |

Es decir, el término más raro (`ave`) recibe el mayor IDF y los términos con TF = 0 tienen peso 0.

In [2]:
# TF: matriz (documentos x términos) con el conteo f(t,d)
def tf(docs, vocab):
    pos = {w: j for j, w in enumerate(vocab)}
    M = np.zeros((len(docs), len(vocab)))
    for i, d in enumerate(docs):
        for w in d:
            if w in pos: M[i, pos[w]] += 1
    return M

def df(TF):      return (TF > 0).sum(axis=0)          # cada documento cuenta una vez por término
def idf(DF, N):  return np.log(N / DF) + 1            # IDF = ln(N/DF) + 1
def tfidf(TF, IDF): return TF * IDF                   # w = TF * IDF

# Prueba con un mini-corpus de 3 documentos
toy = [["gato", "perro", "gato"], ["perro", "pez"], ["gato", "pez", "ave"]]
tv = sorted(set(w for d in toy for w in d))
T = tf(toy, tv); DF_ = df(T); I = idf(DF_, len(toy))
print(pd.DataFrame({"DF": DF_, "IDF": I.round(4), "TF doc0": T[0], "TF-IDF doc0": tfidf(T, I)[0].round(4)}, index=tv))

       DF     IDF  TF doc0  TF-IDF doc0
ave     1  2.0986      0.0       0.0000
gato    2  1.4055      2.0       2.8109
perro   2  1.4055      1.0       1.4055
pez     2  1.4055      0.0       0.0000


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

#### Resultado esperado — Sección 3

- Matriz TF-IDF de forma **(18, tamaño del vocabulario)**.
- **IDF mínimo = 1.0** (términos presentes en los 18 documentos, ln(1)+1) e **IDF máximo ≈ 3.8904** (términos en un solo documento, ln(18)+1).
- Los 6 mayores pesos de *Moby Dick* serán términos propios del libro y muy frecuentes en él: se espera ver `whale` y `ahab` entre los primeros, junto con nombres de personajes (`stubb`, `queequeg`, `starbuck`) y `sperm`.

In [3]:
TF  = tf(docs, vocab)                 # frecuencias
DF  = df(TF)                         # frecuencia documental
IDF = idf(DF, len(docs))             # IDF del corpus
W   = tfidf(TF, IDF)                 # matriz documento-término TF-IDF

print("Forma de la matriz:", W.shape)
print("IDF mín/máx:", IDF.min().round(4), IDF.max().round(4), "| ln(18)+1 =", round(np.log(18) + 1, 4))
k = ids.index("melville-moby_dick.txt")
top6 = np.argsort(-W[k])[:6]         # los 6 pesos más altos de un documento
print(pd.DataFrame({"término": [vocab[j] for j in top6], "TF": TF[k, top6], "DF": DF[top6], "TF-IDF": W[k, top6].round(2)}))

Forma de la matriz: (18, 41071)
IDF mín/máx: 1.0 3.8904 | ln(18)+1 = 3.8904
    término      TF  DF   TF-IDF
0     whale  1226.0   6  2572.90
1      ahab   511.0   2  1633.78
2     stubb   257.0   1   999.83
3  queequeg   252.0   1   980.37
4     sperm   244.0   2   780.12
5  starbuck   198.0   1   770.29


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

#### Resultado esperado — Sección 4

- El orden de columnas de `TfidfVectorizer` coincidirá con el de mi vocabulario (**True**), porque ambos ordenan alfabéticamente.
- Con `smooth_idf=False`, `norm=None` y los mismos tokens, ambas matrices deben ser idénticas salvo error de punto flotante: **máxima diferencia absoluta ≈ 0** (del orden de 1e-12 o menor).

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# analyzer=lambda d: d  -> recibe exactamente los mismos tokens ya procesados
vec = TfidfVectorizer(analyzer=lambda d: d, use_idf=True, smooth_idf=False, sublinear_tf=False, norm=None)
X = vec.fit_transform(docs).toarray()

print("Mismo orden de columnas:", list(vec.get_feature_names_out()) == vocab)
print("Máxima diferencia absoluta:", np.abs(W - X).max())

Mismo orden de columnas: True
Máxima diferencia absoluta: 0.0


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

#### Resultado esperado — Sección 5 (consultas y coseno)

Qué espera encontrar el usuario en cada consulta:

| Consulta | Necesidad de información | Documento que se espera |
|---|---|---|
| Q1 `white whale captain ahab sea ship` | Novela de caza de ballenas y viajes marítimos | Moby Dick |
| Q2 `mrs miss sister marriage fortune` | Novelas de costumbres sobre matrimonio y familia | Novelas de Austen (Emma, Sense and Sensibility, Persuasion) |
| Q3 `satan angels heaven fall` | Poema épico sobre la caída de los ángeles | Paradise Lost |
| Q4 `rabbit queen garden tea` | Cuento infantil fantástico con animales y una reina | Alice's Adventures in Wonderland |

Resultado esperado: la diferencia máxima entre mi coseno explícito y `cosine_similarity` de scikit-learn será **≈ 0 (< 1e-12)**.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Cuatro consultas con necesidades de información distintas
consultas = {
    "Q1": "white whale captain ahab sea ship",
    "Q2": "mrs miss sister marriage fortune",
    "Q3": "satan angels heaven fall",
    "Q4": "rabbit queen garden tea"}

def vec_consulta(q):                                  # vector TF-IDF con vocabulario e IDF del corpus
    return vec.transform([prep(q)]).toarray()[0]      # transform: NO se recalcula el IDF

def coseno(q, d):                                     # q·d / (||q|| ||d||) con producto punto explícito
    return np.dot(q, d) / (np.sqrt(np.dot(q, q)) * np.sqrt(np.dot(d, d)))

Q = {n: vec_consulta(t) for n, t in consultas.items()}
dif = max(abs(coseno(Q[n], W[i]) - cosine_similarity([Q[n]], [W[i]])[0, 0]) for n in Q for i in range(len(docs)))
print("Máx. diferencia explícito vs sklearn (4 consultas x 18 documentos):", dif)

Máx. diferencia explícito vs sklearn (4 consultas x 18 documentos): 5.551115123125783e-16


#### Resultado esperado — desglose Q1 vs. Moby Dick

- Términos compartidos: los que aparezcan en la consulta y en el libro (`whale`, `ahab`, `captain`, `ship`, `sea`, `white`); el mayor producto parcial lo aportará `whale` o `ahab`, por su TF muy alto en el libro.
- El producto punto será positivo y grande; las normas, ||q|| de pocas unidades y ||d|| de varios miles (los pesos TF-IDF de un libro largo son grandes).
- **Similitud alta respecto al resto del corpus (≈ 0.5–0.6)**, siempre dentro de [0, 1] porque los pesos TF-IDF no son negativos.

In [6]:
# Desglose para la combinación Q1 - Moby Dick
q, d = Q["Q1"], W[ids.index("melville-moby_dick.txt")]
comp = np.where((q > 0) & (d > 0))[0]                 # términos compartidos
print(pd.DataFrame({"término": [vocab[j] for j in comp], "q_i": q[comp].round(3), "d_i": d[comp].round(3),
                    "producto parcial": (q[comp] * d[comp]).round(2)}))
print("Producto punto:", round(np.dot(q, d), 2), "| ||q||:", round(np.linalg.norm(q), 2), "| ||d||:", round(np.linalg.norm(d), 2))
print("Similitud:", round(coseno(q, d), 4))

   término    q_i       d_i  producto parcial
0     ahab  3.197  1633.782           5223.57
1  captain  1.693   557.045            943.16
2      sea  1.118   508.591            568.49
3     ship  1.325   686.569            909.99
4    whale  2.099  2572.899           5399.52
5    white  1.118   314.097            351.09
Producto punto: 13395.82 | ||q||: 4.66 | ||d||: 4938.8
Similitud: 0.5816


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

#### Resultado esperado — Sección 6 (rankings top-5)

- **Q1:** 1.º *Moby Dick* con puntuación claramente mayor (≈ 0.5–0.6); el resto, por debajo de 0.1.
- **Q2:** 1.º–3.º, las tres novelas de Austen con puntuaciones parecidas (≈ 0.17–0.20), seguidas de *The Parent's Assistant* (Edgeworth).
- **Q3:** 1.º *Paradise Lost* (≈ 0.2); el resto, cercano a 0.
- **Q4:** 1.º *Alice's Adventures in Wonderland* (≈ 0.1).
- Las versiones explícita y de scikit-learn coinciden (el `assert` con tolerancia 1e-9 no falla) y los fragmentos mostrarán el término con mayor producto parcial en contexto.

In [7]:
def fragmento(i, q):                                  # contexto del término con mayor producto parcial
    t = vocab[np.argmax(q * W[i])]
    m = re.search(r"\b" + t + r"\b", raw[i], re.I)
    return " ".join(raw[i][max(0, m.start() - 60): m.end() + 60].split()) if m else raw[i][:120]

for n, texto in consultas.items():
    s_exp = np.array([coseno(Q[n], W[i]) for i in range(len(docs))])      # versión explícita
    s_lib = cosine_similarity([Q[n]], W)[0]                               # versión sklearn
    assert np.allclose(s_exp, s_lib, atol=1e-9)                           # tolerancia = 1e-9
    top = np.argsort(-s_exp)[:5]
    print(f"\n{n}: {texto}")
    display(pd.DataFrame({"Posición": range(1, 6), "Identificador": [ids[i] for i in top], "Título": [tit[i] for i in top],
                          "Puntuación": s_exp[top].round(4), "Fragmento": [fragmento(i, Q[n]) for i in top]}))


Q1: white whale captain ahab sea ship


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick,0.5816,"in hand to school others, and to teach them by..."
1,2,austen-persuasion.txt,Persuasion,0.0772,"e of Monkford, however suspicious appearances ..."
2,3,whitman-leaves.txt,Leaves of Grass,0.0594,above all promote brave soldiers. } In Cabin'd...
3,4,chesterton-brown.txt,The Wisdom of Father Brown,0.0340,"st and specialist in certain moral disorders, ..."
4,5,bryant-stories.txt,Stories to Tell to Children,0.0285,icked him out again. HOW BROTHER RABBIT FOOLED...



Q2: mrs miss sister marriage fortune


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,austen-sense.txt,Sense and Sensibility,0.1955,nt to them all increased. The constant attenti...
1,2,austen-emma.txt,Emma,0.1952,Emma was aware that great must be the differen...
2,3,austen-persuasion.txt,Persuasion,0.1717,ry had acquired a little artificial importance...
3,4,edgeworth-parents.txt,The Parent's Assistant,0.1263,ld for yourself. You will carry this letter to...
4,5,chesterton-brown.txt,The Wisdom of Father Brown,0.0260,"a pocketful of money, but nobody knows what hi..."



Q3: satan angels heaven fall


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,milton-paradise.txt,Paradise Lost,0.2175,"nce, And justify the ways of God to men. Say f..."
1,2,blake-poems.txt,Poems (Blake),0.0340,became an infant small; Infant smiles are his ...
2,3,whitman-leaves.txt,Leaves of Grass,0.0183,"ever be any more perfection than there is now,..."
3,4,bible-kjv.txt,The King James Bible,0.0180,ses: Called Genesis 1:1 In the beginning God c...
4,5,chesterton-ball.txt,The Ball and the Cross,0.0131,"as spoken, the bottom has fallen out of the Un..."



Q4: rabbit queen garden tea


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,carroll-alice.txt,Alice's Adventures in Wonderland,0.0938,s in Wonderland by Lewis Carroll 1865] CHAPTER...
1,2,bryant-stories.txt,Stories to Tell to Children,0.0301,1918] TWO LITTLE RIDDLES IN RHYME There's a ga...
2,3,chesterton-brown.txt,The Wisdom of Father Brown,0.0237,g softly across the sandy fields and through t...
3,4,blake-poems.txt,Poems (Blake),0.0222,nd seek for her Maker meek; And the desert wil...
4,5,chesterton-ball.txt,The Ball and the Cross,0.0149,hough he had fished him up with a lasso out of...
